# 05 Summary Plots And Interpretation

This notebook combines the stage CSVs under the new reframing:

- **ML2R** is the canonical paper-style shrinkage metric.
- **raw Frobenius ratios** diagnose raw under-scaling.
- **masked ratios** diagnose CI attenuation.
- **output / Jacobian ratios** diagnose functional contraction.


In [1]:
from pathlib import Path
import sys

if (Path.cwd() / 'koko_notebooks').exists():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = Path.cwd().resolve().parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from koko_notebooks.shrinkage_analysis.shrinkage_analysis_common import (
    OUTPUTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
    batched_expected_masked_weight,
    batched_expected_train_mask_weight,
    collect_ci_outputs,
    component_strengths,
    discover_exp07_analysis_jsons,
    ensure_outputs_dir,
    exhaustive_binary_probe_batch,
    latest_result_per_run,
    layer_weight_metric_row,
    load_component_model_for_checkpoint,
    save_dataframe,
    save_json,
    sampled_probe_batch,
    select_consistent_replicate,
    singleton_probe_batch,
)
from koko_notebooks.shrinkage_analysis.publication_plots import (
    ARCH_COLORS,
    LAYER_COLORS,
    architecture_comparison_plot,
    heatmap,
    histogram_triptych,
    line_plot_by_group,
    line_plot_by_layer,
    multi_metric_panel_by_group,
    multi_metric_panel_by_layer,
    parse_vector_column,
    save_figure,
    setup_publication_style,
    singular_value_trajectory_plot,
)

ensure_outputs_dir()
setup_publication_style()
OUTPUTS_DIR


/root/spd_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/workspace/spd/important_outputs/shrinkage_analysis_rep1')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

raw_delta_df = pd.read_csv(OUTPUTS_DIR / 'csv' / 'raw_delta_metrics.csv')
ci_mask_df = pd.read_csv(OUTPUTS_DIR / 'csv' / 'ci_mask_metrics.csv')
functional_df = pd.read_csv(OUTPUTS_DIR / 'csv' / 'functional_metrics.csv')
matching_df = pd.read_csv(OUTPUTS_DIR / 'csv' / 'matching_metrics.csv')

SHRINKAGE_MMCS_THRESHOLD = 0.95
SHRINKAGE_ML2R_THRESHOLDS = [0.95, 0.90, 0.80]

raw_delta_df.head()


,layer_name,target_fro_norm,raw_fro_norm,delta_fro_norm,raw_fro_ratio,delta_fro_ratio,raw_spectral_ratio,faithfulness_mse,target_singular_values,raw_singular_values,run_name,depth,architecture,replicate,checkpoint_step,top1_component_strength_frac,top3_component_strength_frac,raw_under_scaling_gap,delta_support_gap
0,linear1,2.532642,2.539120,0.012483,1.002558,0.004929,1.003195,0.000016,"[1.7944810390472412, 1.7872087955474854]","[1.8002140522003174, 1.7906317710876465]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,0.197995,0.590869,-0.002558,0.004929
1,linear2,2.532642,2.539120,0.012483,1.002558,0.004929,1.003195,0.000016,"[1.7944811582565308, 1.7872086763381958]","[1.8002140522003174, 1.7906321287155151]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,5000,0.197995,0.590869,-0.002558,0.004929
2,linear1,2.532642,2.534972,0.007271,1.000920,0.002871,0.999237,0.000005,"[1.7944810390472412, 1.7872087955474854]","[1.793110966682434, 1.7918813228607178]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,0.197649,0.591620,-0.000920,0.002871
3,linear2,2.532642,2.534972,0.007271,1.000920,0.002871,0.999236,0.000005,"[1.7944811582565308, 1.7872086763381958]","[1.7931108474731445, 1.7918813228607178]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,10000,0.197649,0.591620,-0.000920,0.002871
4,linear1,2.532642,2.536268,0.017241,1.001431,0.006807,1.002247,0.000030,"[1.7944810390472412, 1.7872087955474854]","[1.798512578010559, 1.7882969379425049]",exp_07_tms_5_2_2layer_tied_rep1,2,tied,1,15000,0.198032,0.592059,-0.001431,0.006807


In [3]:
layer_name = 'linear1'

final_raw_df = raw_delta_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)
final_ci_df = ci_mask_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)
final_functional_df = functional_df.sort_values('checkpoint_step').groupby('run_name', as_index=False).tail(1)
final_matching_df = matching_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name'], as_index=False).tail(1)

layer1_raw_df = final_raw_df[final_raw_df['layer_name'] == layer_name].copy()
layer1_ci_df = final_ci_df[final_ci_df['layer_name'] == layer_name].copy()
layer1_matching_df = final_matching_df[final_matching_df['layer_name'] == layer_name].copy()

layer1_summary_df = (
    layer1_raw_df[['run_name', 'depth', 'architecture', 'raw_fro_ratio', 'raw_under_scaling_gap', 'delta_fro_ratio']]
    .merge(layer1_ci_df[['run_name', 'expected_mask_fro_ratio_mean', 'mask_attenuation_gap']], on='run_name', how='inner')
    .merge(final_functional_df[['run_name', 'raw_output_norm_ratio_mean', 'raw_output_contraction_gap', 'raw_jacobian_fro_ratio_mean', 'raw_jacobian_contraction_gap']], on='run_name', how='inner')
    .merge(layer1_matching_df[['run_name', 'mmcs', 'ml2r', 'ml2r_shrinkage_gap', 'paper_shrinkage_flag_0p95', 'paper_shrinkage_flag_0p90']], on='run_name', how='inner')
)
layer1_summary_df.head()


,run_name,depth,architecture,raw_fro_ratio,raw_under_scaling_gap,delta_fro_ratio,expected_mask_fro_ratio_mean,mask_attenuation_gap,raw_output_norm_ratio_mean,raw_output_contraction_gap,raw_jacobian_fro_ratio_mean,raw_jacobian_contraction_gap,mmcs,ml2r,ml2r_shrinkage_gap,paper_shrinkage_flag_0p95,paper_shrinkage_flag_0p90
0,exp_07_tms_5_2_6layer_untied_rep1,6,untied,0.994323,0.005677,0.214001,0.616866,0.377457,9.959018e-01,4.098177e-03,1.003445e+00,-3.445179e-03,0.973549,0.701883,0.298117,1.0,1.0
1,exp_07_tms_5_2_2layer_tied_rep1,2,tied,1.003234,-0.003234,0.003267,0.633007,0.370227,9.456625e-01,5.433750e-02,1.006478e+00,-6.478271e-03,0.999969,0.990876,0.009124,0.0,0.0
2,exp_07_tms_5_2_6layer_tied_rep1,6,tied,1.010643,-0.010643,0.073592,0.628551,0.382093,4.898750e+09,-4.898750e+09,4.131181e+11,-4.131181e+11,0.993437,9.063803,-8.063803,0.0,0.0
3,exp_07_tms_5_2_5layer_untied_rep1,5,untied,1.013333,-0.013333,0.014546,0.626427,0.386906,1.009548e+00,-9.548187e-03,9.189905e-01,8.100951e-02,0.982994,1.054954,-0.054954,0.0,0.0
4,exp_07_tms_5_2_5layer_tied_rep1,5,tied,0.998046,0.001954,0.013238,0.697052,0.300994,9.362853e-01,6.371468e-02,9.989738e-01,1.026241e-03,0.984641,0.890970,0.109030,1.0,1.0


In [4]:
layer1_depth_mean_df = (
    layer1_summary_df.groupby(['depth', 'architecture'], as_index=False)
    .agg(
        raw_fro_ratio=('raw_fro_ratio', 'mean'),
        raw_under_scaling_gap=('raw_under_scaling_gap', 'mean'),
        delta_fro_ratio=('delta_fro_ratio', 'mean'),
        expected_mask_fro_ratio_mean=('expected_mask_fro_ratio_mean', 'mean'),
        mask_attenuation_gap=('mask_attenuation_gap', 'mean'),
        raw_output_norm_ratio_mean=('raw_output_norm_ratio_mean', 'mean'),
        raw_output_contraction_gap=('raw_output_contraction_gap', 'mean'),
        raw_jacobian_fro_ratio_mean=('raw_jacobian_fro_ratio_mean', 'mean'),
        raw_jacobian_contraction_gap=('raw_jacobian_contraction_gap', 'mean'),
        mmcs=('mmcs', 'mean'),
        ml2r=('ml2r', 'mean'),
        ml2r_shrinkage_gap=('ml2r_shrinkage_gap', 'mean'),
        paper_shrinkage_flag_0p95=('paper_shrinkage_flag_0p95', 'mean'),
    )
)

plot_manifest = {}
for metric, ylabel, stem, hline_at_one in [
    ('raw_fro_ratio', 'Layer-1 raw Fro ratio', 'layer1_raw_ratio_vs_depth', True),
    ('raw_under_scaling_gap', 'Layer-1 raw under-scaling gap', 'layer1_raw_under_scaling_vs_depth', False),
    ('mask_attenuation_gap', 'Layer-1 CI attenuation gap', 'layer1_ci_attenuation_vs_depth', False),
    ('raw_output_contraction_gap', 'Layer-1 output contraction gap', 'layer1_output_contraction_vs_depth', False),
    ('raw_jacobian_contraction_gap', 'Layer-1 Jacobian contraction gap', 'layer1_jacobian_contraction_vs_depth', False),
    ('ml2r', 'Layer-1 ML2R', 'layer1_ml2r_vs_depth', True),
    ('ml2r_shrinkage_gap', 'Layer-1 ML2R shrinkage gap', 'layer1_ml2r_gap_vs_depth', False),
    ('paper_shrinkage_flag_0p95', 'Layer-1 paper shrinkage rate (MMCS>=0.95, ML2R<0.95)', 'layer1_paper_shrinkage_rate_vs_depth', False),
]:
    plot_manifest[stem] = architecture_comparison_plot(
        df=layer1_depth_mean_df,
        x_col='depth',
        y_col=metric,
        title=ylabel,
        ylabel=ylabel,
        subdir='summary',
        stem=stem,
        hline_at_one=hline_at_one,
    )
len(plot_manifest)


8

In [5]:
raw_under_scaling_onset_rows = []
for (run_name, layer_name), group in raw_delta_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name']):
    group = group.sort_values('checkpoint_step')
    architecture = group['architecture'].iloc[0]
    depth = int(group['depth'].iloc[0])
    for threshold in [0.95, 0.90, 0.80]:
        below = group[group['raw_fro_ratio'] < threshold]
        raw_under_scaling_onset_rows.append(
            {
                'run_name': run_name,
                'architecture': architecture,
                'depth': depth,
                'layer_name': layer_name,
                'threshold': threshold,
                'first_checkpoint_below_threshold': None if below.empty else int(below['checkpoint_step'].iloc[0]),
            }
        )
raw_under_scaling_onset_df = pd.DataFrame(raw_under_scaling_onset_rows)
save_dataframe(raw_under_scaling_onset_df, 'csv/raw_under_scaling_onset.csv')

ml2r_shrinkage_onset_rows = []
for (run_name, layer_name), group in matching_df.sort_values('checkpoint_step').groupby(['run_name', 'layer_name']):
    group = group.sort_values('checkpoint_step')
    architecture = group['architecture'].iloc[0]
    depth = int(group['depth'].iloc[0])
    for threshold in SHRINKAGE_ML2R_THRESHOLDS:
        flagged = group[(group['mmcs'] >= SHRINKAGE_MMCS_THRESHOLD) & (group['ml2r'] < threshold)]
        ml2r_shrinkage_onset_rows.append(
            {
                'run_name': run_name,
                'architecture': architecture,
                'depth': depth,
                'layer_name': layer_name,
                'threshold_mmcs': SHRINKAGE_MMCS_THRESHOLD,
                'threshold_ml2r': threshold,
                'first_checkpoint_flagged': None if flagged.empty else int(flagged['checkpoint_step'].iloc[0]),
            }
        )
ml2r_shrinkage_onset_df = pd.DataFrame(ml2r_shrinkage_onset_rows)
save_dataframe(ml2r_shrinkage_onset_df, 'csv/ml2r_shrinkage_onset.csv')

raw_under_scaling_onset_df.head(), ml2r_shrinkage_onset_df.head()


(                          run_name architecture  depth layer_name  threshold  \
 0  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1       0.95   
 1  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1       0.90   
 2  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1       0.80   
 3  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear2       0.95   
 4  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear2       0.90   
 
   first_checkpoint_below_threshold  
 0                             None  
 1                             None  
 2                             None  
 3                             None  
 4                             None  ,
                           run_name architecture  depth layer_name  \
 0  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1   
 1  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1   
 2  exp_07_tms_5_2_2layer_tied_rep1         tied      2    linear1   
 3 

In [6]:
onset_mean_df = (
    ml2r_shrinkage_onset_df[ml2r_shrinkage_onset_df['threshold_ml2r'] == 0.90]
    .dropna(subset=['first_checkpoint_flagged'])
    .groupby(['architecture', 'depth', 'layer_name'], as_index=False)
    .agg(first_checkpoint_flagged=('first_checkpoint_flagged', 'mean'))
)
for architecture in ['tied', 'untied']:
    arch_df = onset_mean_df[onset_mean_df['architecture'] == architecture].copy()
    ordered_layers = [layer for layer in LAYER_COLORS if layer in set(arch_df['layer_name'])]
    onset_matrix = arch_df.pivot(index='depth', columns='layer_name', values='first_checkpoint_flagged').reindex(columns=ordered_layers).sort_index()
    if onset_matrix.empty:
        continue
    plot_manifest[f'ml2r_onset_heatmap_{architecture}'] = heatmap(
        matrix=onset_matrix.to_numpy(),
        row_labels=[str(idx) for idx in onset_matrix.index],
        col_labels=list(onset_matrix.columns),
        title=f'Onset of ML2R shrinkage (MMCS>=0.95, ML2R<0.90) | {architecture}',
        colorbar_label='Checkpoint step',
        subdir='summary',
        stem=f'ml2r_onset_heatmap_{architecture}',
        cmap='YlOrBr',
        annotate=True,
        fmt='.0f',
    )
len(plot_manifest)


10

In [7]:
heatmap_metric_specs = [
    ('raw_under_scaling_gap', 'Raw under-scale'),
    ('mask_attenuation_gap', 'CI attenuation'),
    ('raw_output_contraction_gap', 'Output contraction'),
    ('raw_jacobian_contraction_gap', 'Jacobian contraction'),
    ('mmcs', 'MMCS'),
    ('ml2r', 'ML2R'),
    ('ml2r_shrinkage_gap', 'ML2R gap'),
]
for architecture in ['tied', 'untied']:
    arch_df = layer1_depth_mean_df[layer1_depth_mean_df['architecture'] == architecture].copy().sort_values('depth')
    metric_matrix = arch_df[[metric for metric, _ in heatmap_metric_specs]].to_numpy()
    plot_manifest[f'layer1_summary_heatmap_{architecture}'] = heatmap(
        matrix=metric_matrix,
        row_labels=[str(depth) for depth in arch_df['depth']],
        col_labels=[label for _, label in heatmap_metric_specs],
        title=f'Layer-1 reframed summary | {architecture}',
        colorbar_label='Value',
        subdir='summary',
        stem=f'layer1_summary_heatmap_{architecture}',
        vmin=0.0,
        vmax=1.1,
        annotate=True,
    )

fig, axes = plt.subplots(1, 4, figsize=(19, 4.5), constrained_layout=True)
triangle_metrics = [
    ('raw_under_scaling_gap', 'Raw under-scaling'),
    ('mask_attenuation_gap', 'CI attenuation'),
    ('raw_output_contraction_gap', 'Functional contraction'),
    ('ml2r_shrinkage_gap', 'ML2R shrinkage'),
]
tied_df = layer1_depth_mean_df[layer1_depth_mean_df['architecture'] == 'tied'].sort_values('depth')
untied_df = layer1_depth_mean_df[layer1_depth_mean_df['architecture'] == 'untied'].sort_values('depth')
for ax, (metric, title) in zip(axes, triangle_metrics, strict=True):
    ax.plot(tied_df['depth'], tied_df[metric], marker='o', linewidth=2.2, color=ARCH_COLORS['tied'], label='tied')
    ax.plot(untied_df['depth'], untied_df[metric], marker='o', linewidth=2.2, color=ARCH_COLORS['untied'], label='untied')
    ax.set_xlabel('Depth')
    ax.set_title(title)
axes[0].set_ylabel('Gap / value')
axes[-1].legend(frameon=False)
plot_manifest['reframed_summary_panel'] = save_figure(fig, subdir='summary', stem='reframed_summary_panel')

save_json(plot_manifest, 'plots/summary/manifest.json')
plot_manifest


{'layer1_raw_ratio_vs_depth': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_raw_ratio_vs_depth.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_raw_ratio_vs_depth.pdf'},
 'layer1_raw_under_scaling_vs_depth': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_raw_under_scaling_vs_depth.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_raw_under_scaling_vs_depth.pdf'},
 'layer1_ci_attenuation_vs_depth': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_ci_attenuation_vs_depth.png',
  'pdf': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_ci_attenuation_vs_depth.pdf'},
 'layer1_output_contraction_vs_depth': {'png': '/workspace/spd/important_outputs/shrinkage_analysis_rep1/plots/summary/layer1_output_contraction_vs_depth.png',
  'pdf': '/workspace/spd/importan